# KXHIGHLAX Implied Spot Analysis

Back out the market-implied expected LA high temperature from Kalshi binary/range option prices.

**Market structure:** each event lists "between" (range bucket, e.g. `78-79` inclusive both ends),
"greater" (strict upper tail, e.g. `> 85`), and "less" (strict lower tail, e.g. `< 78`) markets.

**Method:** each market's own model-implied probability under `T ~ N(μ, σ²)` is fit directly against
its own `[bid, ask]` band (no cross-strike cumulative sum) -- see `black_scholes.fit_normal_to_bucket_bid_ask`.
Settlement (NWS daily high) is a whole-degree integer and every strike is inclusive of the named
integer(s), so each boundary gets a ±0.5 continuity correction before comparing against the continuous
Normal -- see `black_scholes._continuity_corrected_bounds`.

In [ ]:
import sys
import os
import importlib
import random
_NB_DIR = os.path.dirname(os.path.abspath('__file__'))
sys.path.insert(0, _NB_DIR)
sys.path.insert(0, os.path.join(_NB_DIR, "..", "kalshi_infra"))

import datetime
from datetime import timezone, timedelta

import api_functions
import black_scholes
import predexon_functions
import plotting
importlib.reload(api_functions)
importlib.reload(black_scholes)
importlib.reload(predexon_functions)
importlib.reload(plotting)

from api_functions import event_ticker_from_date
from black_scholes import compute_implied_spot_timeseries_cached
from predexon_functions import load_predexon_long_df
from plotting import LA_TZ

In [ ]:
def day_window_ts(date):
    """
    Return (start_ts, end_ts) covering the full market lifetime in PT.
    Uses PT midnight as the day boundary since KXHIGHLAX resolves on the PT calendar day.
    Window: previous PT midnight → next PT midnight (48h).
    """
    event_start_pt = LA_TZ.localize(datetime.datetime(date.year, date.month, date.day))
    start = event_start_pt - timedelta(days=1)
    end = event_start_pt + timedelta(days=1)
    now = int(datetime.datetime.now(timezone.utc).timestamp())
    return int(start.timestamp()), min(int(end.timestamp()), now)


def fetch_day_timeseries(date, freq='1s', max_lookback=None):
    """
    Implied spot/sigma for one day, on a `freq` grid over its full market
    lifetime. All the actual work -- caching, and only calling Predexon
    (a live API call + a multi-MB raw-tick parquet read) on an actual cache
    miss -- lives in compute_implied_spot_timeseries_cached; this is just
    wiring the day's window and event ticker into it. Defined here (before
    both the winddown section and the 3-day sample section below) since both
    need it.
    """
    event_ticker = event_ticker_from_date(date)
    start_ts, end_ts = day_window_ts(date)
    ts_df = compute_implied_spot_timeseries_cached(
        lambda: load_predexon_long_df(event_ticker),
        start_ts, end_ts, event_ticker, freq=freq, max_lookback=max_lookback,
    )
    print(f'{event_ticker} ({date}): {len(ts_df)} time points ({freq} grid)')
    return event_ticker, ts_df, date


KALSHI_WS_CACHE_DIR = os.path.join(_NB_DIR, 'data', 'implied_ts_kalshi_ws')
# Widened from compute_implied_spot_timeseries's own 90s default (tuned for
# Predexon's much denser tick stream) -- the WS collector only logs a new
# quote when the top of book actually changes, so a quiet book can
# legitimately go several minutes with no new tick without being "stale."
# Same value kalshi_vol_trading/mispricing_analysis.ipynb uses. This is the
# FIT's own per-market staleness tolerance -- a separate concept from the
# day-level "did the whole feed go quiet" flag computed in the Day selection
# section below (see plotting.kalshi_ws_feed_gap_mask's docstring for why
# those two can't share a threshold).
KALSHI_WS_FIT_STALENESS_THRESHOLD = '10min'


def fetch_day_timeseries_kalshi_ws(date, freq='1min'):
    """
    Implied spot/sigma for one day, fit ENTIRELY off Kalshi's own live L2
    order book (plotting.load_kalshi_ws_long_df) instead of Predexon -- see
    the Day selection section below for why. Zero live network calls either
    way (a local parquet read on a cache miss, a local parquet read on a
    cache hit). kalshi_backup is passed as an explicit EMPTY DataFrame so
    compute_implied_spot_timeseries's own internal staleness/mismatch
    fallback -- normally a live Kalshi 1-min-candle API call -- never fires;
    long_df is already Kalshi's own data, so there's nothing to fall back
    to. Same pattern kalshi_vol_trading/mispricing_analysis.ipynb uses.
    """
    event_ticker = event_ticker_from_date(date)
    start_ts, end_ts = day_window_ts(date)
    ts_df = compute_implied_spot_timeseries_cached(
        lambda: plotting.load_kalshi_ws_long_df(event_ticker),
        start_ts, end_ts, event_ticker, freq=freq,
        cache_dir=KALSHI_WS_CACHE_DIR, fit_smile=True,
        staleness_threshold=KALSHI_WS_FIT_STALENESS_THRESHOLD, kalshi_backup=plotting._EMPTY_KALSHI_BACKUP,
    )
    print(f'{event_ticker} ({date}): {len(ts_df)} time points ({freq} grid, kalshi_ws)')
    return event_ticker, ts_df, date

## Day selection

Two separate day pools, now drawn from two different sources:

- **`SPOT_PLOT_DAYS`**: every locally available `kalshi_ws` day for KXHIGHLAX
  (`plotting.list_available_kalshi_ws_days` -- a plain local directory scan, no network call) --
  Kalshi's own live L2 order book, not Predexon (see `fetch_day_timeseries_kalshi_ws` /
  `plotting.load_kalshi_ws_long_df`'s docstrings for why: real crossed-book defects were confirmed
  in the Predexon-sourced feed). This only covers however far back the WS collector has actually
  been running -- currently ~2 weeks (16 days) -- far short of Predexon's multi-month archive
  `WINDDOWN_DAYS` below still draws on.

  Each day gets a `plotting.kalshi_ws_feed_gap_fraction` -- the fraction of the day where the
  WHOLE kalshi_ws feed went quiet (no listed market updated AT ALL, not just one thin one -- see
  that function's docstring for why a naive per-market staleness check way overflags a change-
  only-logged feed like this one, and why these gaps line up with actual WS-collector restarts in
  `kalshi_infra/logs/`). If at least `MIN_CLEAN_KALSHI_WS_DAYS` days are fully clean (0% feed gap),
  the partially-stale days are dropped outright and `SPOT_PLOT_DAYS` becomes just the clean ones.
  Otherwise -- the actual case right now, only ~3 of 16 days are fully clean -- every locally
  available day is kept, and each day's own stale stretches are shaded directly on the plot below
  instead (amber, via `feed_gap_by_ticker`), alongside the existing (red) lambda_min-rejected-fit
  shading that already applied regardless of data source.

- **`WINDDOWN_DAYS`**: unchanged -- the 15 cached days immediately BEFORE `SPOT_PLOT_DAYS`'
  earliest day, still sourced from Predexon's much longer archive (`black_scholes.
  list_cached_days`), since the winddown curve needs far more history than ~2 weeks of kalshi_ws
  can supply. `_build_winddown_per_day` already excludes a day's own gaps from its contribution to
  the aggregate mean, so this pool still doesn't need its own gap filter.

In [ ]:
WINDDOWN_POOL_SIZE = 15
KALSHI_WS_FEED_GAP_THRESHOLD = '15min'  # day-level "did the WHOLE feed go quiet" threshold -- see
                                         # plotting.kalshi_ws_feed_gap_mask's docstring
MIN_CLEAN_KALSHI_WS_DAYS = 8  # "enough days with no stale data" to drop the partially-stale ones
                               # outright rather than keep-and-flag them (see markdown above) --
                               # set to roughly half of what list_available_kalshi_ws_days actually
                               # returns right now (16), so this is a real bar, not a rubber stamp

CACHED_DAYS = black_scholes.list_cached_days(freq='1min')
print(f'{len(CACHED_DAYS)} Predexon-cached days locally, {CACHED_DAYS[0]} .. {CACHED_DAYS[-1]}')

ALL_KALSHI_WS_DAYS = plotting.list_available_kalshi_ws_days('KXHIGHLAX')
print(f'\n{len(ALL_KALSHI_WS_DAYS)} kalshi_ws days locally, {ALL_KALSHI_WS_DAYS[0]} .. {ALL_KALSHI_WS_DAYS[-1]}')

kalshi_ws_day_results_1min = [fetch_day_timeseries_kalshi_ws(d, freq='1min') for d in ALL_KALSHI_WS_DAYS]

kalshi_ws_gap_fracs = {}
for event_ticker, df, d in kalshi_ws_day_results_1min:
    kalshi_ws_gap_fracs[d] = 1.0 if df.empty else plotting.kalshi_ws_feed_gap_fraction(
        df, d, event_ticker, gap_threshold=KALSHI_WS_FEED_GAP_THRESHOLD)

print(f'\nkalshi_ws feed-gap fraction per day (collector-outage stretches, {KALSHI_WS_FEED_GAP_THRESHOLD} threshold):')
for d in ALL_KALSHI_WS_DAYS:
    frac = kalshi_ws_gap_fracs[d]
    print(f'  {d}: {frac:.1%}' + ('  <- clean' if frac == 0.0 else ''))

CLEAN_KALSHI_WS_DAYS = [d for d in ALL_KALSHI_WS_DAYS if kalshi_ws_gap_fracs[d] == 0.0]
if len(CLEAN_KALSHI_WS_DAYS) >= MIN_CLEAN_KALSHI_WS_DAYS:
    SPOT_PLOT_DAYS = CLEAN_KALSHI_WS_DAYS
    print(f'\n{len(CLEAN_KALSHI_WS_DAYS)} fully clean days >= {MIN_CLEAN_KALSHI_WS_DAYS} needed -- '
          f'dropping the {len(ALL_KALSHI_WS_DAYS) - len(CLEAN_KALSHI_WS_DAYS)} day(s) with any stale stretch entirely.')
else:
    SPOT_PLOT_DAYS = ALL_KALSHI_WS_DAYS
    print(f'\nOnly {len(CLEAN_KALSHI_WS_DAYS)} fully clean day(s), short of the {MIN_CLEAN_KALSHI_WS_DAYS} needed to '
          f'drop the rest -- keeping all {len(ALL_KALSHI_WS_DAYS)} days, with each day\'s own stale '
          f'stretches shaded on the plot below instead.')
print('SPOT_PLOT_DAYS:', SPOT_PLOT_DAYS)

WINDDOWN_DAYS = [d for d in CACHED_DAYS if d < SPOT_PLOT_DAYS[0]][-WINDDOWN_POOL_SIZE:]
print('\nWINDDOWN_DAYS (Predexon, unaffected by the kalshi_ws switch above):', WINDDOWN_DAYS)

## Realized-variance winddown shape -- 15 days before the spot/vol sample

`variance_till_expiry` necessarily decreases to 0 by expiry, but not necessarily linearly in
calendar time -- if it isn't, predicting remaining realized variance (to compare against implied)
needs this empirical winddown shape rather than the linear-in-time assumption `iv_scaled_to_day`
makes. Fit on `WINDDOWN_DAYS` (see the Day selection section above) -- a separate, strictly OLDER
15-day pool than `SPOT_PLOT_DAYS`, so the curve reweighting that section's implied vol isn't fit
on (part of) the same days it gets applied to. Computed here, before the spot/vol section below,
so its result can directly reweight that section's implied-vol panel (see `winddown_curve` passed
into `plot_implied_spot_history`).

Prints goodness-of-fit tables for both mean curves this fits (vertical and landmark -- see
`plot_variance_winddown`'s docstring for what each means); the **landmark** curve is the one
actually used to reweight implied vol below, since it preserves the timing of when variance
typically resolves rather than smearing it out.

In [ ]:
winddown_day_results = [fetch_day_timeseries(d, freq='1min') for d in WINDDOWN_DAYS]
vert_grid, vert_pct, winddown_grid, winddown_pct = plotting.plot_variance_winddown(winddown_day_results)

## Diagnostic – naive vs. noise-corrected total realized variance

`variance_till_expiry`'s naive sum-of-squared-diffs is sampling-rate SENSITIVE in both directions:
LM-fit tick-to-tick wobble inflates it at fine grids (a genuinely noisy series' realized variance
should INCREASE, not converge, as you sample finer), while a genuinely smooth real move can get
misrepresented as one big instantaneous jump at a COARSE grid (confirmed on 2026-07-22: a real,
continuous ~1.5° move over ~60 seconds reads as a single 1min-to-1min jump, inflating the coarse
total). `plotting.two_scale_realized_variance` (Zhang/Mykland/Aït-Sahalia TSRV) corrects for the
first (noise inflating the fine grid) by combining a fast-scale and an averaged slow-scale
estimate off the SAME 1s series -- see its docstring for the full derivation.

Compares, per cached day: the naive 1min total against TSRV (computed off a freshly-fetched 1s
series) and against implied variance -- TSRV should pull days whose naive numbers were
noise-driven outliers back toward the implied-variance ratio, while leaving genuinely-low-realized
days (real market convergence, not measurement noise) roughly where they were.

**Fixed -- all three numbers now share the same "day start" anchor.** This originally compared
each of naive_1min/TSRV/iv2_start using ITS OWN default starting point: naive_1min and iv2_start
from `day_results`' row 0 (the start of the full 48h FETCH window, a day earlier than the actual
settlement day), while TSRV was computed over the 1s series' entire span. Investigating an
apparent "implied is overpredicting, and it's too systematic to be real mispricing" pattern on
2026-07-29 traced back to exactly this: the window's row 0 is the event's very first tick, right
as it opened trading -- confirmed directly (`n_two_sided=2`, `implied_sigma=1.94` at the first
valid fit vs. 5-6 markets and a steady ~1.1-1.7 within minutes) -- so `iv2_start` was reading an
inflated, thin-liquidity opening price, and `naive_1min` was picking up whatever noisy move
happened during that same thin stretch, a full day before the actual calendar day even begins.
Both numbers were distorted by the same root cause in the same direction, which is exactly what
made a comparison bug look like a bug in the realized-variance math itself (it isn't one --
verified independently by hand: `variance_till_expiry`'s row-0 total matched a from-scratch
sum-of-squared-diffs to full float precision).

`plotting.realized_vs_implied_at_day_start` fixes this: naive_1min, TSRV, and iv2_start are all
now anchored at `plotting.pt_midnight_utc(date)` -- the actual start of the calendar day the event
resolves on -- rather than each series' own row 0. Confirmed on 2026-07-29: iv2_start drops from
3.77 (thin opening tick) to 1.17 (properly-liquid reading), and naive_1min/TSRV -- now both
measuring the SAME from-day-start remainder -- land much closer to each other (0.455 vs. 0.435)
than the old mismatched-anchor comparison ever showed.

In [ ]:
rv_comparison_rows = []
for ticker, df_1min, date in winddown_day_results:
    if df_1min.empty:
        continue
    _, df_1s, _ = fetch_day_timeseries(date, freq='1s')
    naive_1min, tsrv, iv2_start = plotting.realized_vs_implied_at_day_start(df_1min, df_1s, date)
    rv_comparison_rows.append((ticker, date, naive_1min, tsrv, iv2_start))

print(f"\n{'ticker':<24}{'naive_1min':>11}{'TSRV':>9}{'iv2_start':>11}{'ratio_1min':>11}{'ratio_tsrv':>11}")
for ticker, date, naive_1min, tsrv, iv2_start in rv_comparison_rows:
    ratio_1min = naive_1min / iv2_start if iv2_start else float('nan')
    ratio_tsrv = tsrv / iv2_start if iv2_start else float('nan')
    print(f'{ticker:<24}{naive_1min:>11.3f}{tsrv:>9.3f}{iv2_start:>11.3f}{ratio_1min:>11.2f}{ratio_tsrv:>11.2f}')

## Historical implied spot – kalshi_ws days (~2 weeks)

Every day in `SPOT_PLOT_DAYS` (see the Day selection section above), fit directly off Kalshi's own
live L2 order book rather than Predexon. Fetched at a single `1min` grid -- unlike the Predexon-
sourced diagnostics elsewhere in this notebook, there's no value in a finer grid here: the WS
collector only logs a new quote when a market's own top-of-book actually changes, so between two
real updates a finer eval grid would just re-fit the exact same inputs to the exact same output, at
up to 60x the compute cost for zero new information. (The noise-correction TSRV line other
Predexon-sourced day_results get from a paired 1s/1min fetch is skipped here for the same reason --
it corrects for tick-to-tick sampling noise in a dense continuous stream, which doesn't exist in a
change-only-logged feed.)

Both raw ticks (the WS collector's own cached parquet) and the fitted implied_spot/sigma series are
cached locally, so this is fast for any day already fetched -- `SPOT_PLOT_DAYS` was already fetched
at `1min` above while computing each day's feed-gap fraction, so this section just reuses that.

In [ ]:
# Reuse the 1min fetch already done for every kalshi_ws day above (see the
# Day selection section) rather than re-fetching -- SPOT_PLOT_DAYS is a
# subset (or all) of ALL_KALSHI_WS_DAYS, and compute_implied_spot_
# timeseries_cached would just hit its own on-disk cache anyway, so this
# only saves the redundant lookup, not real work.
_spot_plot_day_set = set(SPOT_PLOT_DAYS)
day_results = [r for r in kalshi_ws_day_results_1min if r[2] in _spot_plot_day_set]
print(f'{len(day_results)} day(s) selected for the historical spot/vol plot.')

In [ ]:
plotting.plot_implied_spot_history(
    day_results,
    winddown_curve=(winddown_grid, winddown_pct),
    # Amber shading for stretches where the WHOLE kalshi_ws feed went quiet
    # (see the Day selection section / plotting.kalshi_ws_feed_gap_mask) --
    # a separate, visually distinct flag from the existing red lambda_min-
    # rejected-fit shading, which still applies regardless of data source.
    feed_gap_by_ticker={
        ticker: plotting.kalshi_ws_feed_gap_mask(ticker, df['time'], gap_threshold=KALSHI_WS_FEED_GAP_THRESHOLD)
        for ticker, df, date in day_results if not df.empty
    },
    # One row, not the default 5-per-row wrap: same per-day panel sizing
    # either way (col_width/total_height_frac are unchanged), this just lets
    # the figure's total width grow with the day count rather than wrapping
    # into a second (equally-sized) row.
    days_per_row=len(day_results),
)

## Diagnostic – 10 worst-fitting snapshots

For the 10 snapshots across EVERY LOCALLY CACHED day (via `black_scholes.list_cached_days` --
excludes the still-accruing current month, same convention as the skew diagnostic below) where the
symmetric-normal (or local-vol-smile) model itself fit WORST -- ranked by `implied_band_cost`, the
fit's own sum of squared band-violation residuals at its best (mu, sigma), i.e. how much every
listed market's own `[bid, ask]` band still had to be violated even at the optimum -- show that
snapshot's own raw bid/ask alongside the fitted model's probability curve. This is a different
failure mode than the old biggest-jump diagnostic: a big, cleanly-explained move ranks high there
but not here, while a small move the model still can't reconcile across its listed markets ranks
high here but not there.

Previously scoped to just `SPOT_PLOT_DAYS` (the 10 most recent days) -- widened to every cached
day since the worst incidents turned out to be concentrated in FEB/MAR, not recent data at all
(band_cost up to 0.67 there vs. under 0.05 anywhere in July).

Every market -- `'between'` included, not just `<F`/`>F` tail markets -- is plotted at its own TRUE
strike (no repositioning), displaced vertically from the reference curve's own height there by
(quoted bid/ask - the model's own price for that market) rather than as a raw probability value: a
quote that matches the model sits ON the curve, one pricing more than the model sits above it, less
sits below (see `_render_snapshot_panel`'s docstring). This sidesteps the shape mismatch between a
tail market's true cumulative price and the window-shaped reference curve without repositioning
onto a borrowed bucket the way an earlier version of this diagnostic did.

In [ ]:
WORST_FIT_DAYS = black_scholes.list_cached_days(freq='1s', on_or_before=datetime.date(2026, 7, 31))
print(f'{len(WORST_FIT_DAYS)} cached days (Jul 2026 and earlier) for the worst-fits pass')

worst_fit_day_results = [fetch_day_timeseries(d) for d in WORST_FIT_DAYS]
worst_fits = plotting.plot_worst_fits(worst_fit_day_results, top_n=10)

## Diagnostic – symmetric-normal skew check, by month

The fit assumes `T ~ N(μ, σ²)`, a symmetric distribution -- if the market's actual (informed)
view of the LA high has persistent skew (e.g. a marine-layer ceiling capping upside more often
than cold snaps get capped on the downside, or the reverse in heat-wave season), a symmetric
model will show the SAME side as structurally cheap and the other as structurally expensive on
every single day. That's model misspecification masquerading as edge, not real mispricing.

For a sample of timestamps across every LOCALLY CACHED day from July 2026 and earlier (via
`black_scholes.list_cached_days` -- excludes August so this never depends on a still-accruing/
partial current month), plots each market's `model_prob - mid` against `z = (strike location - μ)
/ σ` (its distance from the fitted center, in the fitted model's own σ units), one panel per
calendar month. A flat binned mean around 0 means the symmetric assumption looks fine; a
systematic slope means the model is structurally biased toward one side. Splitting by month
checks whether that bias is a stable, seasonal-weather-driven effect (shows up consistently
across months with different regimes) vs. an artifact of a handful of unusual days dominating a
pooled sample -- see `plot_vol_skew_diagnostic`'s docstring.

This pulls in far more days than the other sections (every cached day, not just the small pools
selected above) for statistical power, so the underlying per-snapshot gather is cached to parquet
(see `use_cache`/`_skew_cache_key`) -- the first run will be slow, reruns after that load straight
from disk.

**Update -- the deep-hot-tail bias is real, and now (partially) corrected.** Running this
originally showed a persistent, same-sign bias in the deepest delta bin (>=~0.92) in EVERY single
month Jan-Jul, regardless of season -- exactly the "structural model misspecification" pattern this
diagnostic exists to catch. A day-level and raw-quote deep dive (see session notes) found:
- A small (~1%) minority of the worst individual cases were pure data artifacts (frozen/stale
  quotes disconnected from that day's own moving fit, or effectively-unquoted markets with a
  $0.01/$0.98-style non-spread) -- real, but not the bulk of the effect.
- The bulk of the bias came from genuinely liquid, tight-spread (median 1¢), well-collateralized
  markets, spread across hundreds of distinct tickers and most of the 187 cached days -- i.e. the
  market's real, persistent view, not noise.

Two fixes were tried and rejected before landing on the current one:
- A single global **skew** parameter (splitting the fit into a two-piece normal with a different
  scale above/below μ) reshaped the ENTIRE upper (or lower) half of the distribution, not just the
  actually-biased tail -- it improved the deep bin but visibly distorted mid-delta bins that were
  already fine. Removed.
- A **fixed-delta pillar smile** (e.g. literal 15d/50d/85d points) turned out to be too fragile: a
  typical event only lists ~6 markets, and a check found 25-33% of snapshots had ZERO listed
  market anywhere near a fixed 15d/85d target.

What's live now: `black_scholes.fit_local_sigma_smile` builds a **local-volatility smile** from
WHATEVER 'less'/'greater' wing markets are actually listed that day (~98% of days list exactly
one of each), each priced off its own interpolated point on a curve that's pinned FLAT at
`implied_sigma` out to `|z| <= 1.3` (matching where the diagnostic itself shows the bias actually
starts) and only bends toward the wing's own implied value beyond that -- so a deep-tail
correction can't bleed into bins that were already fine. Validated against the full 187-day
sample: every month's hot-tail bin improved (2-25%, largest in the months with the smallest
starting bias), while every ATM bin moved by <0.001. `fit_smile=True` is now baked into
`regen_cache.py`'s regen, so every cached day carries the `implied_wing_*` columns this needs --
pass `use_smile=True` below to reprice off the smile instead of the plain symmetric model.

In [ ]:
SKEW_DAYS = black_scholes.list_cached_days(freq='1min', on_or_before=datetime.date(2026, 7, 31))
print(f'{len(SKEW_DAYS)} cached days (Jul 2026 and earlier) for the skew diagnostic')

skew_day_results = [fetch_day_timeseries(d, freq='1min') for d in SKEW_DAYS]
skew_diag = plotting.plot_vol_skew_diagnostic(skew_day_results, use_smile=True)